# 03 — Classification & Clustering AnalysisThis notebook explores:1. Binary classification of peptides into high/low absorbance groups2. Unsupervised clustering to discover structural groupings3. Cluster-based absorbance profiling for new sequences

In [ ]:
import sys, ossys.path.insert(0, os.path.abspath('..'))import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_scorefrom sklearn.preprocessing import StandardScalerfrom sklearn.ensemble import RandomForestClassifier, VotingClassifierfrom sklearn.linear_model import LogisticRegressionfrom sklearn.svm import SVCfrom sklearn.cluster import KMeansfrom sklearn.decomposition import PCAfrom sklearn.metrics import classification_report, f1_scorefrom src.feature_extraction import extract_all_featuresfrom src.visualization import plot_pca_clusters%matplotlib inline

## 1. Load Data & Create Binary Labels

In [ ]:
DATA_DIR = os.path.join('..', 'data')df_model = pd.read_csv(os.path.join(DATA_DIR, 'processed_features.csv'))# Binary label: above median = high (1), below = low (0)median_abs = df_model['mean_abs'].median()df_model['abs_bin'] = (df_model['mean_abs'] > median_abs).astype(int)print(f'Median absorbance: {median_abs:.4f}')print(f'Class balance:\n{df_model["abs_bin"].value_counts(normalize=True)}')

In [ ]:
top_features = [    'negative_charge_ratio', 'aa_percent_E', 'flex_mean', 'aa_percent_D',    'charged_ratio', 'instability_index', 'n_term_polar', 'turn_frac', 'mol_weight',]X = df_model[top_features]y = df_model['abs_bin']X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)scaler = StandardScaler()X_train_scaled = scaler.fit_transform(X_train)X_test_scaled = scaler.transform(X_test)

## 2. Train Classifiers

In [ ]:
classifiers = {    'LogisticRegression': LogisticRegression(),    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),    'SVM': SVC(probability=True),}for name, clf in classifiers.items():    clf.fit(X_train_scaled, y_train)    y_pred = clf.predict(X_test_scaled)    f1 = f1_score(y_test, y_pred)    print(f'\n{name} — F1: {f1:.3f}')    print(classification_report(y_test, y_pred))

In [ ]:
# Cross-validated F1 for Random Forestrf_clf = RandomForestClassifier(n_estimators=100, random_state=42)cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)scores = cross_val_score(rf_clf, X, y, cv=cv, scoring='f1')print(f'CV F1 scores: {scores}')print(f'Mean F1: {scores.mean():.3f} (+/- {scores.std():.3f})')

## 3. Ensemble Voting Classifier

In [ ]:
ensemble = VotingClassifier(    estimators=[        ('lr', LogisticRegression()),        ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),        ('svm', SVC(probability=True)),    ],    voting='soft',)ensemble.fit(X_train_scaled, y_train)y_pred_ens = ensemble.predict(X_test_scaled)print(f'Ensemble F1: {f1_score(y_test, y_pred_ens):.3f}')print(classification_report(y_test, y_pred_ens))

## 4. Feature Importance (Classification)

In [ ]:
rf_full = RandomForestClassifier(n_estimators=100, random_state=42)rf_full.fit(X, y)importances = pd.Series(rf_full.feature_importances_, index=top_features)importances = importances.sort_values(ascending=True)plt.figure(figsize=(8, 5))importances.plot(kind='barh')plt.title('Feature Importances (Random Forest Classifier)')plt.xlabel('Importance')plt.tight_layout()plt.show()

## 5. KMeans Clustering

In [ ]:
X_clust = df_model[top_features].valuesabsorbance = df_model['mean_abs'].values# Cluster into 3 groupskmeans = KMeans(n_clusters=3, random_state=42, n_init=10)labels = kmeans.fit_predict(X_clust)fig = plot_pca_clusters(X_clust, labels, absorbance)plt.show()

In [ ]:
# Cluster absorbance statisticsdf_model['cluster'] = labelscluster_stats = df_model.groupby('cluster')['mean_abs'].agg(['mean', 'std', 'min', 'max', 'count'])print('Absorbance statistics per cluster:')cluster_stats

## 6. Cluster-Based Prediction for New Sequences

In [ ]:
def predict_cluster_profile(sequence, kmeans_model, cluster_stats_df, top_features_list):    """Predict which cluster a new peptide belongs to and return absorbance stats."""    feats = extract_all_features(sequence)    if feats is None:        return None    X_new = pd.DataFrame([{f: feats.get(f, 0) for f in top_features_list}])    cluster_id = kmeans_model.predict(X_new)[0]    stats = cluster_stats_df.loc[cluster_id]    return {        'sequence': sequence,        'cluster': cluster_id,        'mean_abs': stats['mean'],        'std_abs': stats['std'],        'min_abs': stats['min'],        'max_abs': stats['max'],    }# Test with new sequencesnew_seqs = ['SYENSHSQAINVDRT', 'DEHRNNQSSSTAIVY', 'DDDDDDDDDDDDDDD']for seq in new_seqs:    result = predict_cluster_profile(seq, kmeans, cluster_stats, top_features)    print(result)